In [6]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# =========================
# Config
# =========================
SEED = 42
np.random.seed(SEED)

base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"

file_names = [
#     "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
#     "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
#     "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
    "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
]

target_vars = ["Jsc", "Voc", "FF", "PCEmax"]
# 55行目付近（TARGET_VARS の後あたりに追加）
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'Var.1'
]
today = datetime.now().strftime("%Y%m%d")

output_dir = f"./{today}_Rebuilt_12Files_RboristLikeRF_Train_CV10OOF_OOD"
os.makedirs(output_dir, exist_ok=True)


# =========================
# Helpers
# =========================
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_r2(y_true, y_pred) -> float:
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return float("nan")

def create_ood_split(df: pd.DataFrame, feature_cols, ood_ratio=0.1):
    """重心からの距離が大きい 10% を OOD にする"""
    X = df[feature_cols].to_numpy()
    centroid = np.mean(X, axis=0)
    distances = np.linalg.norm(X - centroid, axis=1)
    num_ood = max(1, int(len(df) * ood_ratio))
    ood_idx = np.argsort(distances)[-num_ood:]
    df_train = df.drop(df.index[ood_idx])
    df_ood = df.iloc[ood_idx]
    return df_train, df_ood

def remove_zero_variance(df: pd.DataFrame, cols):
    v = df[cols].var(axis=0, skipna=True)
    keep = v[(v > 0) & (~v.isna())].index.tolist()
    return keep

def remove_collinear_features(df: pd.DataFrame, cols, threshold=0.99999):
    """相関がほぼ1の重複特徴を削除"""
    if len(cols) < 2:
        return cols
    corr = df[cols].corr().abs().fillna(0.0)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    if to_drop:
        print(f"  [Cleaning] Removed {len(to_drop)} duplicated features (corr>{threshold})")
    return [c for c in cols if c not in to_drop]

def overfit_flags(train_r2, cv_r2, train_rmse, cv_rmse,
                  r2_gap_thr=0.10, rmse_gap_ratio_thr=0.10):
    """Train vs CV10_OOF の差で Overfit を判定（Rコードと同じルール）"""
    if np.isnan(train_r2) or np.isnan(cv_r2) or np.isnan(train_rmse) or np.isnan(cv_rmse):
        return False, "OK"
    delta_r2 = train_r2 - cv_r2
    delta_rmse = cv_rmse - train_rmse
    flag = (delta_r2 >= r2_gap_thr) or (
        (train_rmse > 0) and (delta_rmse >= rmse_gap_ratio_thr * abs(train_rmse))
    )
    label = "Overfit_suspected" if flag else "OK"
    return flag, label, float(delta_r2), float(delta_rmse)

def build_oof_predictions_rf(best_params, X_train, y_train, n_splits=10):
    """ベストパラメータで RandomForest の CV10 OOF 予測を作る"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.full(shape=(X_train.shape[0],), fill_value=np.nan, dtype=float)

    for fold, (tr, va) in enumerate(kf.split(X_train), start=1):
        model = RandomForestRegressor(
            random_state=SEED,
            n_jobs=-1,
            **best_params
        )
        model.fit(X_train[tr], y_train.iloc[tr])
        oof[va] = model.predict(X_train[va])

    return oof


# =========================
# Main
# =========================
summary_rows = []
importance_rows = []

print(f"[INFO] Output dir: {output_dir}")
print(f"[INFO] Input base path: {base_path}")

for file in file_names:
    in_path = os.path.join(base_path, file)
    if not os.path.exists(in_path):
        print(f"[WARN] Skipping: {file} (Not found)")
        continue

    print(f"\n=== Processing Rborist-like RF: {file} ===")
    try:
        df_raw = pd.read_csv(in_path, index_col=0)
        df_raw = df_raw.drop(columns=["X"], errors="ignore")
    except Exception as e:
        print(f"[ERROR] read_csv failed: {file} | {e}")
        continue

    # 数値列のみ & 欠損除去
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()
    if df_num.shape[0] < 20:
        print(f"  [WARN] Too few rows after dropna: {df_num.shape[0]}")
        continue

    for target in target_vars:
        if target not in df_num.columns:
            continue

        # ✅ features = 「4つの目的変数をすべて除外」
        feature_cols = [c for c in df_num.columns if c not in target_vars]
        # ★修正：変数を 'file' と 'inappropriate_vars' に変更
        if file.startswith("m_all"):
            feature_cols = [c for c in feature_cols if c not in inappropriate_vars]
            print(f"    [Security] Removed inappropriate vars for {file}")
        if len(feature_cols) == 0:
            print("    [WARN] No features after excluding targets. Skipped.")
            continue
        feature_cols = remove_zero_variance(df_num, feature_cols)
        feature_cols = remove_collinear_features(df_num, feature_cols, threshold=0.99999)

        if len(feature_cols) == 0 or df_num.shape[0] < 20:
            continue

        # OOD split
        df_train, df_ood = create_ood_split(df_num, feature_cols, ood_ratio=0.1)

        # Scaling (-1, 1)  ※ R と同じ発想
        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_train = scaler.fit_transform(df_train[feature_cols])
        y_train = df_train[target]
        X_ood = scaler.transform(df_ood[feature_cols])
        y_ood = df_ood[target]

        # -------------------------
        # Hyperparameter tuning (Rborist 相当のランダムフォレスト)
        # -------------------------
        base_rf = RandomForestRegressor(
            random_state=SEED,
            n_jobs=-1
        )

        param_grid = {
            "n_estimators": [200, 500],
            "max_depth": [None, 20, 40],
            "min_samples_leaf": [1, 3, 5],
            "max_features": ["sqrt", 0.3, 0.5]
        }

        cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
        grid = GridSearchCV(
            estimator=base_rf,
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=cv,
            n_jobs=-1,
            refit=True
        )

        try:
            grid.fit(X_train, y_train)
        except Exception as e:
            print(f"  [ERROR] GridSearch failed Target={target}: {e}")
            continue

        best_model = grid.best_estimator_
        best_params = grid.best_params_

        # -------------------------
        # Train / CV10_OOF / OOD
        # -------------------------
        # Train
        pred_train = best_model.predict(X_train)
        train_r2 = safe_r2(y_train, pred_train)
        train_rmse = rmse(y_train, pred_train)
        train_mae = float(mean_absolute_error(y_train, pred_train))
        train_rpd = float(np.std(y_train) / train_rmse) if train_rmse != 0 else 0.0

        # CV10_OOF
        pred_oof = build_oof_predictions_rf(best_params, X_train, y_train, n_splits=10)
        cv_r2 = safe_r2(y_train, pred_oof)
        cv_rmse = rmse(y_train, pred_oof)
        cv_mae = float(mean_absolute_error(y_train, pred_oof))
        cv_rpd = float(np.std(y_train) / cv_rmse) if cv_rmse != 0 else 0.0

        # OOD
        pred_ood = best_model.predict(X_ood)
        ood_r2 = safe_r2(y_ood, pred_ood)
        ood_rmse = rmse(y_ood, pred_ood)
        ood_mae = float(mean_absolute_error(y_ood, pred_ood))
        ood_rpd = float(np.std(y_ood) / ood_rmse) if ood_rmse != 0 else 0.0

        overfit_flag, overfit_label, delta_r2, delta_rmse = overfit_flags(
            train_r2, cv_r2, train_rmse, cv_rmse,
            r2_gap_thr=0.10,
            rmse_gap_ratio_thr=0.10
        )

        summary_rows.append({
            "File": file,
            "Target": target,
            "Train_R2": train_r2,
            "Train_RMSE": train_rmse,
            "Train_MAE": train_mae,
            "Train_RPD": train_rpd,
            "CV10_R2": cv_r2,
            "CV10_RMSE": cv_rmse,
            "CV10_MAE": cv_mae,
            "CV10_RPD": cv_rpd,
            "OOD_R2": ood_r2,
            "OOD_RMSE": ood_rmse,
            "OOD_MAE": ood_mae,
            "OOD_RPD": ood_rpd,
            "n_samples": int(df_num.shape[0]),
            "n_features": int(len(feature_cols)),
            "Best_Params": json.dumps(best_params, ensure_ascii=False),
            "Overfit_Flag": bool(overfit_flag),
            "Overfit_Label": overfit_label,
            "Delta_R2_TrainMinusCV10": delta_r2,
            "Delta_RMSE_CV10MinusTrain": delta_rmse
        })

        print(
            f"  Target={target} | "
            f"Train_R2={train_r2:.3f} | CV10_R2={cv_r2:.3f} | OOD_R2={ood_r2:.3f} | {overfit_label}"
        )

        # -------------------------
        # Save model bundle
        # -------------------------
        bundle = {
            "model": best_model,
            "scaler": scaler,
            "feature_cols": feature_cols,
            "target": target,
            "file": file,
            "best_params": best_params
        }
        model_name = f"fixed_{today}_RboristLikeRF_model_{file.replace('.csv','')}_{target}.joblib"
        joblib.dump(bundle, os.path.join(output_dir, model_name))

        # -------------------------
        # Save prediction CSV (Train / CV10_OOF / OOD_Test) identical columns
        # -------------------------
        df_pred_train = pd.DataFrame({
            "SampleID": df_train.index.astype(str),
            "Observed": y_train.to_numpy(),
            "Predicted": pred_train.astype(float),
            "Type": ["Train"] * len(y_train)
        })
        df_pred_oof = pd.DataFrame({
            "SampleID": df_train.index.astype(str),
            "Observed": y_train.to_numpy(),
            "Predicted": pred_oof.astype(float),
            "Type": ["CV10_OOF"] * len(y_train)
        })
        df_pred_ood = pd.DataFrame({
            "SampleID": df_ood.index.astype(str),
            "Observed": y_ood.to_numpy(),
            "Predicted": pred_ood.astype(float),
            "Type": ["OOD_Test"] * len(y_ood)
        })

        df_pred_all = pd.concat(
            [df_pred_train, df_pred_oof, df_pred_ood],
            axis=0,
            ignore_index=True
        )
        pred_name = f"fixed_{today}_RboristLikeRF_{file.replace('.csv','')}_{target}_pred.csv"
        df_pred_all.to_csv(os.path.join(output_dir, pred_name), index=False)

        # -------------------------
        # Feature importance (Gini-based)
        # -------------------------
        importances = best_model.feature_importances_
        order = np.argsort(importances)[::-1]
        for idx in order:
            imp_val = float(importances[idx])
            if imp_val <= 0:
                continue
            importance_rows.append({
                "File": file,
                "Target": target,
                "Feature": feature_cols[idx],
                "Importance_Mean": imp_val,
                "Importance_Std": float("nan")  # RFの標準 importance なので std は無し
            })

# =========================
# Save summary / importance
# =========================
df_sum = pd.DataFrame(summary_rows)
sum_path = os.path.join(output_dir, f"fixed_{today}_RboristLikeRF_summary.csv")
df_sum.to_csv(sum_path, index=False)

df_imp = pd.DataFrame(importance_rows)
imp_path = os.path.join(output_dir, f"fixed_{today}_RboristLikeRF_feature_importance.csv")
df_imp.to_csv(imp_path, index=False)

print("\n[DONE]")
print(f"Summary : {sum_path}")
print(f"Imp     : {imp_path}")


[INFO] Output dir: ./20260112_Rebuilt_12Files_RboristLikeRF_Train_CV10OOF_OOD
[INFO] Input base path: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/

=== Processing Rborist-like RF: m_all.csv ===
    [Security] Removed inappropriate vars for m_all.csv
  Target=Jsc | Train_R2=0.949 | CV10_R2=0.624 | OOD_R2=0.345 | Overfit_suspected
    [Security] Removed inappropriate vars for m_all.csv
  Target=Voc | Train_R2=0.938 | CV10_R2=0.556 | OOD_R2=0.224 | Overfit_suspected
    [Security] Removed inappropriate vars for m_all.csv
  Target=FF | Train_R2=0.932 | CV10_R2=0.499 | OOD_R2=0.431 | Overfit_suspected
    [Security] Removed inappropriate vars for m_all.csv
  Target=PCEmax | Train_R2=0.954 | CV10_R2=0.655 | OOD_R2=0.368 | Overfit_suspected

=== Processing Rborist-like RF: m_all_OH_rebuilt.csv ===
    [Security] Removed inappropriate vars for m_all_OH_rebuilt.csv
  [Cleaning] Removed 3 duplicated